# 06 · Analizador de trades

**Valor de un trade para un equipo:** es el cambio en los puntos esperados de su **alineación óptima** (`lineup.py`) desde ahora hasta el final de la temporada. Se calcula semana a semana, con los byes y la probabilidad de lesión, y las semanas de playoffs de la liga pesan ×2. Se evalúa **para los dos equipos** del trade.

Etapas:
- **(a)** proyecciones semana a semana y evaluación de un trade concreto
- **(b)** incertidumbre: Monte Carlo y backtest de horizonte
- **(c)** buscador de trades 1x1 y 2x1 en los que ganan los dos equipos (sección 6)

Código: `src/fantasy_ml/trades.py` · configuración: `config/trades.yaml`.

## 1. Liga: calendario, fecha límite y reglas del roster

Todo se lee de la configuración de la liga en ESPN, no se escribe a mano.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import polars as pl

import numpy as np
from fantasy_ml import data, espn, trades as T

CFG = data.load_config("trades")
PARAMS = data.load_config("model")["params"]
GDL = ZoneInfo("America/Mexico_City")

league = espn.connect(2026)
cal = T.league_calendar(league, CFG["playoff_weight"])
rules = T.league_rules(league)
ME = espn.my_team_id()

days_left = (cal.trade_deadline - datetime.now(timezone.utc)).days
print(f"Temporada regular: semanas {cal.reg_weeks[0]}–{cal.reg_weeks[-1]} · playoffs: semanas {cal.playoff_weeks} (peso ×{cal.playoff_weight:g})")
print(f"Fecha límite de trades: {cal.trade_deadline.astimezone(GDL):%d-%m-%Y %H:%M} (Guadalajara) · faltan {days_left} días")
print(f"Semanas que se valoran: {cal.weeks[0]}–{cal.weeks[-1]} ({len(cal.weeks)})")
print(f"Slots titulares: { {k: v for k, v in rules['slots'].items() if k not in ('BE', 'IR')} }")
print(f"Roster: {CFG['max_active_roster']} activos + IR · límites por posición: {rules['position_limits']}")
if days_left < 0:
    print("⚠ La fecha límite ya pasó: los trades ya no se pueden hacer.")

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(24)

## 2. Probabilidad de jugar

El modelo predice los puntos **si el jugador juega**. En un horizonte de 15 semanas eso no basta, así que cada semana se multiplica por la probabilidad de jugar. Esa probabilidad sale de una **cadena de disponibilidad** (sección 2.3):
- **Punto de partida:** el estado actual de ESPN (OUT 0, doubtful 0.25, questionable 0.75; IR: 4 semanas fuera como mínimo).
- **A largo plazo:** converge a la tasa de partidos jugados de cada jugador.

### Qué cuenta como partido perdido
Se usa el estado semanal de nflverse en los partidos de su equipo (temporada regular):
- **Jugó:** hay registro de snaps o estadísticas.
- **Perdido:** estaba **inactivo (INA) o en reserva/lesionados (RES)**. Es decir, lesión o inactividad.
- **Excluido:** estaba activo sin jugar (suplente o banqueado). Es cuestión de rol, no de salud, así que no cuenta ni como jugado ni como perdido.

Tampoco cuentan la **semana 18** (descansos de fin de temporada; la liga termina en la 17), los byes ni las semanas en que el jugador no estaba en el equipo.

In [ ]:
src = data.load_sources()
rw_all = pl.concat([data.load_rosters_weekly(s) for s in (2023, 2024, 2025, cal.season)], how="diagonal_relaxed")
state = T.season_state(src, rw_all.filter(pl.col("season") == cal.season), data.load_config("scoring"), cal.season, cal.current_week)
history = T.availability_history(rw_all.filter((pl.col("season") < cal.season) | (pl.col("week") < cal.current_week)),
                                 state["base"], state["points_k"], state["team_games"])
print("Partidos jugador-equipo 2023–hoy:", dict(history["outcome"].value_counts().rows()))
rates = T.availability_rates(history, state["base"], CFG["availability"])
rates.with_columns(pl.col("play_rate").round(3))

### Historial de cada jugador (shrinkage)

La tasa de cada jugador combina su historial con la de su posición: `(jugados + κ·tasa_posición) / (jugados + perdidos + κ)`.

**κ** es cuántos partidos "vale" la tasa de la posición. Lo elijo por **capacidad predictiva**: con el historial hasta la temporada anterior predigo la disponibilidad real de 2024 y 2025 (jugadores relevantes, semanas 5–17) y me quedo con el κ de menor error. No uso el método de momentos porque las lesiones vienen en rachas (una rotura son muchos partidos perdidos seguidos), y eso infla la diferencia aparente entre jugadores.

In [ ]:
kappa, kgrid = T.shrinkage_strength(history, state["base"], rates, CFG["availability"])
print(f"κ elegido: {kappa:.0f} partidos")
kgrid.with_columns(pl.col("rmse").round(5),
                   mejora_vs_solo_posicion_pct=((kgrid["rmse"][-1] - pl.col("rmse")) / kgrid["rmse"][-1] * 100).round(2))

**Resultado:** el historial individual predice **muy poco**. El κ óptimo es alto (el historial pesa poco) y mejora el error menos de un 0.1% frente a usar solo la tasa de la posición. La mayoría de las lesiones son mala suerte y no se repiten. Aun así, el shrinkage separa un poco a los jugadores sanos de los que se lesionan seguido.

También probé separar la tasa por nivel de uso (`xfp`) y no mejoró la predicción de forma apreciable (0.2%). Además hacía que Allen saliera por debajo de Purdy solo por su `xfp`, que mide volumen de juego y no salud.

### 2.3 Las ausencias vienen en rachas: cadena de disponibilidad

Aplicar la tasa de largo plazo a cada semana por separado es un error. Un jugador sano hoy tiene un ~94% de probabilidad de jugar la semana siguiente, no un 84%. Y una ausencia por inactividad suele durar poco, mientras que una en reserva o IR suele durar mucho.

Por eso uso una **cadena de Markov con tres estados**: jugando (P), inactivo (INA, ausencia corta) y reserva o IR (RES, ausencia larga). Las transiciones se estiman con partidos consecutivos de 2023–2025. Para cada jugador escalo la probabilidad de pasar de "jugando" a "ausente" para que su fracción de semanas jugando a largo plazo sea su tasa. El cálculo determinista usa la probabilidad exacta de esta cadena, y el Monte Carlo la simula: los dos son consistentes.

In [ ]:
TM = T.absence_transitions(history, seasons=CFG["availability"]["seasons"])
pl.DataFrame(TM.round(3), schema=[f"→{s}" for s in T.AVAIL_STATES]).insert_column(0, pl.Series("desde", T.AVAIL_STATES))

## 3. Proyección semana a semana

- **Features de forma:** congeladas en su valor actual, que es lo último que se sabe.
- **Contexto del partido de cada semana:** rival, local o visitante, descanso, estadio, y lo que permite hoy la defensa rival a esa posición.
- **Líneas de apuestas:** solo se usan las reales de la semana actual. Para las semanas siguientes se **estiman** con un modelo de fuerza ofensiva y defensiva ajustado a las líneas ya publicadas. Frente a las líneas reales de la semana 4, que ya existen, el error medio es de ~1.2 puntos de *implied total* (correlación 0.89).
- **Comprobación:** en la semana actual, esta proyección coincide exactamente con el registro de predicciones del notebook 04.

In [ ]:
proj = T.add_play_rates(T.project_rest_of_season(state, PARAMS, cal, CFG), history, rates, kappa)
league_proj = T.with_expected_points(proj, T.league_rosters(league), cal, CFG, TM)
fa_proj = T.with_expected_points(proj, T.league_free_agents(league, cal.current_week), cal, CFG, TM).filter(pl.col("proj").is_not_null())
print(f"Agentes libres con proyección (para el nivel de reemplazo): {fa_proj['espn_id'].n_unique()}")

missing = league_proj.filter(pl.col("proj").is_null())["name"].unique().to_list()
print(f"Jugadores con roster en la liga: {league_proj['espn_id'].n_unique()} · sin proyección: {missing or 'ninguno'}")
print(f"Filas jugador-semana: {league_proj.height:,}")

### Probabilidad de jugar de algunos jugadores

In [ ]:
NAMES = ["Josh Allen", "Brock Purdy", "Jayden Daniels", "Jonathan Taylor", "Cam Skattebo", "Nico Collins", "Brandon Aubrey"]
(league_proj.filter(pl.col("name").is_in(NAMES))
    .group_by("name", "position", "injury_status", "prior_rate", "hist_played", "hist_missed", "play_rate")
    .agg(pl.col("p_play").sort_by("week").head(6).round(2).alias("p_jugar_sem_3a8"))
    .with_columns(pl.col("prior_rate", "play_rate").round(3))
    .sort("position", "play_rate", descending=[False, True]))

### Mi roster: puntos esperados por semana (proyección × probabilidad de jugar; `bye` = semana libre)

In [ ]:
mine = league_proj.filter(pl.col("fantasy_team_id") == ME)
wide = (mine.with_columns(cell=pl.when(pl.col("bye")).then(pl.lit("bye")).otherwise(pl.col("exp_points").round(1).cast(pl.Utf8)))
            .pivot(on="week", index=["name", "position"], values="cell", sort_columns=True))
ros = (mine.join(cal.weights(), on="week")
           .group_by("name").agg((pl.col("exp_points") * pl.col("weight")).sum().round(0).alias("ros_modelo"),
                                 (pl.col("espn_points") * pl.col("weight")).sum().round(0).alias("ros_espn"),
                                 pl.col("play_rate").first().round(2).alias("p_jugar")))
wide.join(ros, on="name").sort("ros_modelo", descending=True)

`ros_modelo` y `ros_espn` son totales ponderados (playoffs ×2) hasta la semana 17. El de ESPN es más alto por dos razones: ESPN supone que el jugador juega todos los partidos (solo descuenta byes y lesiones ya conocidas), y el modelo subestima a los titulares a inicio de temporada (notebook 03). Para valorar un trade importan las **diferencias** entre jugadores, no el nivel absoluto.

## 4. Evaluación de un trade concreto

Indica el equipo rival y los jugadores por nombre, tal como aparecen en ESPN. Para cada equipo se muestra:
- **`delta`:** cambio en el valor ponderado de su roster (playoffs ×2), desglosado en temporada regular y playoffs (sin ponderar).
- **`suelta`:** si el equipo queda con más de 15 activos, suelta al jugador que menos valor le aporta.
- **`valido`:** si el trade respeta los límites por posición.
- **Vista de ESPN:** el mismo cálculo con las proyecciones de ESPN, que es lo que ve el otro manager. Sirve para anticipar si aceptaría.

### Nivel de reemplazo
Ningún manager deja un slot vacío ni alinea a alguien peor que el mejor agente libre disponible: ficharía al agente libre. En cada semana:
1. **Los mejores agentes libres de cada posición compiten por todos los slots** con los jugadores del roster. Un slot vacío (bye, OUT, IR), o uno cuyo titular sea peor que el mejor agente libre, lo ocupa ese agente libre. Así, **un jugador vale lo que aporta por encima del mejor agente libre**, y el valor de un roster nunca baja por tener un jugador más.
2. **Titulares con riesgo de no jugar** (probabilidad *p* < 1): aportan `p × su proyección + (1 − p) × el mejor reemplazo`. El reemplazo es el mejor suplente de su banca o agente libre elegible, y cada uno se usa para un solo titular. Es la aproximación en valor esperado; el Monte Carlo de la sección 5 lo simula exactamente.

Una primera versión solo usaba agentes libres para los slots **vacíos**. Eso hacía que quitarle a un equipo un titular peor que el agente libre *aumentara* su valor (por ejemplo, +47.6 al quitarle a Pablo's Perfect Team su D/ST). Con la regla actual comprobé en los 154 jugadores con roster que quitar a cualquiera nunca aumenta el valor.

_La incertidumbre de estas cifras está en la sección 5._

In [ ]:
# ¿En qué equipo está el jugador que quiero? (búsqueda por nombre en todos los rosters)
league_proj.filter(pl.col("name").str.contains("Henderson")).unique("espn_id").select("name", "position", "pro_team", "fantasy_team", "lineup_slot", "injury_status")

In [ ]:
TRADE = {"partner": "David", "give": ["Brock Purdy"], "get": ["TreVeyon Henderson"]}

partner = T.resolve_team(league_proj, TRADE["partner"])
give = T.resolve_players(league_proj, ME, TRADE["give"])
get = T.resolve_players(league_proj, partner, TRADE["get"])

res_model = T.evaluate_trade(league_proj, ME, partner, give, get, rules, cal, CFG, "exp_points", repl=fa_proj)
res_espn = T.evaluate_trade(league_proj, ME, partner, give, get, rules, cal, CFG, "espn_points", repl=fa_proj)
(res_model.select("fantasy_team", "da", "recibe", "suelta", "valor_antes", "valor_despues", "delta", "delta_regular", "delta_playoffs", "valido", "motivo")
          .join(res_espn.select("fantasy_team", delta_segun_espn="delta"), on="fantasy_team"))

In [ ]:
both_win = (res_model["delta"] > 0).all()
espn_other = res_espn.filter(pl.col("fantasy_team_id") == partner)["delta"][0]
print("Según el modelo:", "ganan los dos equipos ✓" if both_win else "no ganan los dos ✗")
print(f"Según ESPN, {TRADE['partner']} {'gana' if espn_other > 0 else 'pierde'} {abs(espn_other):.1f} puntos ponderados "
      f"→ {'probablemente lo aceptaría' if espn_other > 0 else 'probablemente lo rechazaría si decide mirando a ESPN'}")

### Efecto del nivel de reemplazo en este trade

Mismo trade valorado sin agentes libres (solo el roster) y con ellos.

In [ ]:
no_repl = T.evaluate_trade(league_proj, ME, partner, give, get, rules, cal, CFG, "exp_points", repl=None)
(no_repl.select("fantasy_team", sin_reemplazo="delta")
        .join(res_model.select("fantasy_team", con_reemplazo="delta"), on="fantasy_team"))

### Mi alineación semana a semana

Titulares de la alineación óptima antes y después del trade (determinista, con nivel de reemplazo). Solo se listan las semanas en que cambia algún titular. `(agente libre)` indica un slot vacío que se llena con el mejor agente libre.

In [ ]:
before_r, after_r, _, _ = T.rosters_after_trade(league_proj, ME, partner, give, get, rules, cal, CFG, "exp_points", fa_proj)
lu_before = T.weekly_lineups(before_r, rules["slots"], "exp_points", fa_proj)
lu_after = T.weekly_lineups(after_r, rules["slots"], "exp_points", fa_proj)
T.lineup_changes_by_week(lu_before, lu_after)

### Semana 13: descansan Taylor, Tyler Warren, Bateman y los Ravens

In [ ]:
WEEK_FOCUS = 13
byes = league_proj.filter(pl.col("fantasy_team_id") == ME, pl.col("week") == WEEK_FOCUS, pl.col("bye"))["name"].to_list()
print(f"Byes de mi roster en la semana {WEEK_FOCUS}: {byes}")
(lu_before.filter(pl.col("week") == WEEK_FOCUS).select("slot", antes="name", origen_antes="source", pts_antes=pl.col("exp_points").round(1))
    .with_columns(lu_after.filter(pl.col("week") == WEEK_FOCUS).select(despues="name", origen_despues="source",
                                                                        pts_despues=pl.col("exp_points").round(1))))

## 5. Incertidumbre

### 5.1 Backtest de horizonte: ¿cuánto empeora la proyección a más semanas?

Con datos de **2025**, que no se usó para ajustar hiperparámetros, me sitúo en las semanas 3, 6, 9 y 12. En cada una uso **solo lo que se sabía entonces** (`truncate_sources`) y proyecto hasta la semana 17 con el mismo método de producción: features congeladas y líneas estimadas. Después comparo con lo que pasó. Comprobé que a 0 semanas el error es el mismo que el del backtest semanal del notebook 03 (MAE 4.100 contra 4.099), así que no se cuela información del futuro.

Los resultados se guardan en caché. Pon `RECOMPUTE_HORIZON = True` para recalcularlos.

In [ ]:
RECOMPUTE_HORIZON = False
horizon_path = data.DATA_PROC / "horizon_backtest_2025.parquet"
if RECOMPUTE_HORIZON or not horizon_path.exists():
    hb = T.horizon_backtest(src, data.load_rosters_weekly(2025), data.load_config("scoring"), PARAMS, CFG, season=2025)
    hb.write_parquet(horizon_path)
hb = pl.read_parquet(horizon_path)

bucket = pl.col("horizon").cut([0, 2, 5, 9], labels=["0", "1-2", "3-5", "6-9", "10+"], left_closed=False)
(hb.with_columns(semanas_adelante=bucket).group_by("semanas_adelante")
   .agg(pl.len().alias("n"), pl.col("resid").abs().mean().round(2).alias("mae"),
        pl.col("resid").std().round(2).alias("desv_error"), (-pl.col("resid")).mean().round(2).alias("sesgo_pred_menos_real"))
   .sort("semanas_adelante"))

El error crece con el horizonte: de ~5.7 a ~6.6 puntos de desviación (+16%), sin sesgo apreciable. El aumento es moderado porque la forma de un jugador cambia despacio. Lo que más pesa a largo plazo es la disponibilidad, que se modela aparte.

### 5.2 Parámetros del Monte Carlo
- **Error de cada semana:** se muestrea de los **residuos reales** del backtest de horizonte, según posición y quintil de proyección. Así conserva la asimetría de los partidos explosivos y crece con la proyección.
- **Parte persistente del error (ρ):** qué fracción del error se repite semana a semana para el mismo jugador (por ejemplo, un cambio de rol que el modelo tarda en ver). Con ρ = 0, los errores de 15 semanas se cancelarían entre sí más de lo que ocurre en realidad.
- **Factor de horizonte:** la desviación del error a *k* semanas dividida entre la global.

In [ ]:
unc = T.uncertainty_params(hb, TM)
print("ρ (fracción persistente del error):", {k: round(v["rho"], 3) for k, v in unc["pools"].items()})
print("factor de horizonte (0, 1-2, 3-5, 6-9, 10+ semanas):", [round(unc["horizon_factor"][k], 3) for k in sorted(unc["horizon_factor"])])

### 5.3 Simulación

Simulo **2,000 temporadas**. En cada una:
1. **Quién juega:** cada jugador, incluidos los agentes libres, sigue su cadena de disponibilidad semana a semana.
2. **Cuántos puntos hace:** su proyección más un error muestreado. Una parte del error es persistente por jugador y el error crece con el horizonte. Está centrado en la proyección del modelo: el Monte Carlo mide incertidumbre, no recalibra.
3. **Alineación:** cada semana se elige la alineación óptima entre los que juegan, **por proyección y no por resultado** (sin ver el futuro). Los slots vacíos se llenan con el mejor agente libre que juegue. Luego se suman los puntos reales de los titulares, con los playoffs ×2.

Para comparar antes y después del trade se usan **los mismos sorteos**. Así la diferencia es precisa aunque el valor de cada roster tenga mucho ruido.

In [ ]:
N_SIMS = 2000
sim = T.simulate(pl.concat([league_proj, fa_proj], how="diagonal_relaxed"), cal, CFG, unc, n_sims=N_SIMS, seed=0)

# Comprobación: disponibilidad media simulada vs probabilidad del cálculo determinista
both = pl.concat([league_proj, fa_proj], how="diagonal_relaxed").unique(["espn_id", "week"]).filter(~pl.col("bye"))
ix, wx = {e: k for k, e in enumerate(sim.ids)}, {w: k for k, w in enumerate(sim.weeks)}
sim_p = [float(sim.avail[:, ix[e], wx[w]].mean()) for e, w in zip(both["espn_id"], both["week"])]
gap = (pl.Series(sim_p) - both["p_play"]).abs()
print(f"{sim.avail.shape[1]} jugadores × {len(sim.weeks)} semanas × {N_SIMS} simulaciones")
print(f"Disponibilidad simulada vs determinista: diferencia media {gap.mean():.4f}, máxima {gap.max():.3f}")

### 5.4 Mi roster: puntos del resto de la temporada con intervalo del 80%

Puntos reales simulados de cada jugador (solo cuando juega, sin ponderar), sume o no a la alineación. El intervalo refleja el ruido semana a semana, la parte persistente del error y las lesiones.

In [ ]:
mine_ids = league_proj.filter(pl.col("fantasy_team_id") == ME).unique("espn_id").select("espn_id", "name", "position")
rows = []
for r in mine_ids.to_dicts():
    i = sim.index([r["espn_id"]])[0]
    tot = (sim.points[:, i, :] * sim.avail[:, i, :]).sum(axis=1)
    rows.append({"jugador": r["name"], "pos": r["position"], "media": round(float(tot.mean()), 0),
                 "p10": round(float(np.percentile(tot, 10)), 0), "p90": round(float(np.percentile(tot, 90)), 0),
                 "partidos_esperados": round(float(sim.avail[:, i, :].sum(axis=1).mean()), 1)})
pl.DataFrame(rows).sort("media", descending=True)

### 5.5 El trade de ejemplo con incertidumbre

- **`media`:** cambio esperado en el valor ponderado. **`error_mc`** es su error de Monte Carlo, es decir, cuánto cambiaría con otra semilla.
- **`p10` a `p90`:** intervalo del 80% de lo que puede pasar realmente.
- **`prob_gana`:** fracción de temporadas simuladas en que el trade mejora a ese equipo.
- **`determinista`:** la estimación de la sección 4. Difiere algo de la media porque el Monte Carlo elige cada alineación sabiendo quién juega, en lugar de aproximarlo con `p`.

In [ ]:
mc, draws = T.mc_evaluate_trade(sim, league_proj, fa_proj, ME, partner, give, get, rules, cal, CFG, return_draws=True)
(mc.join(res_model.select(pl.col("fantasy_team").str.strip_chars(), determinista="delta"), on="fantasy_team")
   .join(res_espn.select(pl.col("fantasy_team").str.strip_chars(), segun_espn="delta"), on="fantasy_team")
   .select("fantasy_team", "determinista", pl.col("media").round(1), pl.col("error_mc").round(1), "p10", "p50", "p90",
           pl.col("prob_gana").round(2), "segun_espn"))

**Cómo leerlo:** incluso un trade con una media claramente positiva tiene una probabilidad considerable de salir mal, porque 15 semanas de puntos de fantasy tienen mucho ruido. `prob_gana` resume mejor la decisión que la media sola.

### 5.6 Mi cambio semana a semana (Monte Carlo)

Diferencia de puntos reales de mi alineación, después menos antes del trade, en cada semana (sin ponderar). `prob_mejora` y `prob_empeora` no suman 1 porque en muchas simulaciones el trade no cambia nada esa semana: el jugador nuevo no entra a la alineación.

In [ ]:
T.mc_weekly_summary(draws[ME]["weekly"], sim, cal)

## 6. Buscador de trades

Reviso los rosters de los otros 9 equipos y busco trades **1x1 y 2x1, en las dos direcciones** (doy 2 y recibo 1, o doy 1 y recibo 2), en los que **ganan los dos equipos** según el modelo. Solo incluyo QB, RB, WR y TE; K y D/ST casi nunca se intercambian. Hay unos 30,000 posibles, así que lo hago en tres pasos:

1. **Filtro rápido** con valores marginales: cuánto pierde cada equipo si cede a cada jugador y cuánto gana si recibe a cada candidato, menos el jugador que tendría que soltar si su roster queda lleno. Pasan los trades en que ambos ganarían según esa aproximación.
2. **Valor exacto** de los candidatos, con límites de roster y nivel de reemplazo. Se quedan los que dan Δ > 0 para los dos y se quitan los **redundantes**, es decir, los que no mejoran a una versión más simple con el mismo equipo (por ejemplo, "Purdy por Metcalf + Shakir" cuando Shakir se suelta).
3. **Monte Carlo y vista de ESPN** de los 30 mejores, ordenados por el beneficio del equipo que menos gana, porque así salen primero los trades equilibrados, que tienen más probabilidad de aceptarse.

Tarda unos 5–6 minutos.

In [ ]:
found = T.find_trades(league_proj, fa_proj, rules, cal, CFG, ME, sim=sim)
ex = found["exact"]
print(f"Precisión del filtro: corr(aproximado, exacto) = {ex.select(pl.corr('aprox_yo', 'delta_yo')).item():.2f} (yo) · "
      f"{ex.select(pl.corr('aprox_ellos', 'delta_ellos')).item():.2f} (ellos)")

### Mejores propuestas

- **`delta_*`:** determinista.
- **`mc_*`, `p10`/`p90`, `prob_*`:** Monte Carlo (media, intervalo del 80% y probabilidad de ganar).
- **`espn_*`:** el mismo cálculo con las proyecciones de ESPN. **`espn_ellos` negativo** avisa de que el otro manager, si decide mirando a ESPN, probablemente lo rechazaría.
- **`suelto_yo`/`suelta_ellos`:** a quién tendría que soltar cada equipo si su roster queda lleno.

In [ ]:
top = found["top"].with_columns(
    aceptable_segun_espn=pl.col("espn_ellos") >= 0,
    concluyente=(pl.col("prob_gano") >= 0.6) & (pl.col("prob_ganan") >= 0.6))
top.select("equipo", "tipo", "doy", "recibo", "delta_yo", "delta_ellos", "mc_yo", "p10_yo", "p90_yo", "prob_gano",
           "mc_ellos", "prob_ganan", "espn_yo", "espn_ellos", "aceptable_segun_espn", "suelto_yo", "suelta_ellos")

In [ ]:
print(f"Trades en que ganan los dos (sin redundantes): {found['both'].height}")
print(f"De los {top.height} mejores: {top['aceptable_segun_espn'].sum()} también parecen aceptables para el otro equipo según ESPN; "
      f"{top['concluyente'].sum()} tienen probabilidad de ganar ≥ 60% para los dos")
top.filter("aceptable_segun_espn").select("equipo", "doy", "recibo", "mc_yo", "prob_gano", "mc_ellos", "prob_ganan", "espn_ellos").head(10)

**Cómo leerlo:**
- Con nivel de reemplazo, las ganancias de un trade suelen ser pequeñas frente al ruido de una temporada: unos pocos puntos por semana como mucho.
- Una probabilidad de ganar cercana al 50% significa que el trade es casi una moneda al aire, aunque su media sea positiva.
- Los trades más interesantes combinan tres cosas: media positiva para los dos, probabilidad de ganar claramente por encima del 50% y `espn_ellos ≥ 0`, para que el otro manager lo vea con buenos ojos en la app.